# DDRI 무샘플링 학습 실행 노트북

이 노트북은 `06_prediction_training`의 대표 실행 문서다.

원칙:
- 스크립트를 한 번에 호출하는 래퍼로 쓰지 않는다.
- 전처리, 학습, 검증, 재학습, 테스트를 셀 단위로 직접 실행한다.
- 각 단계마다 중간 결과를 표로 확인한다.
- 임포트는 항상 별도 셀로 분리한다.


## 1. 임포트

모든 실행 노트북은 임포트만 담당하는 셀을 따로 둔다.


In [1]:
from pathlib import Path
import importlib.util
import json
import sys

import numpy as np
import pandas as pd


## 1-1. 커널과 의존성 확인

이 셀에서는 현재 노트북 커널이 어떤 Python을 쓰는지와, 학습에 필요한 핵심 패키지가 실제로 import 되는지 먼저 확인한다.


In [2]:
import sys

print('python executable:', sys.executable)
for module_name in ['lightgbm', 'catboost', 'sklearn', 'pandas', 'numpy']:
    try:
        __import__(module_name)
        print(module_name, 'OK')
    except Exception as exc:
        print(module_name, 'MISSING', type(exc).__name__, str(exc))


python executable: /opt/anaconda3/envs/py312/bin/python
lightgbm OK
catboost OK
sklearn OK
pandas OK
numpy OK


## 2. 모듈과 대상 데이터셋 선택

전처리 스크립트와 학습 스크립트의 함수를 직접 불러와 노트북 안에서 단계별로 실행한다.


In [3]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works' / '06_prediction_training'
MODEL_SCRIPT = WORK_DIR / '05_scripts' / '04_ddri_prediction_train_eval.py'
FEATURE_SCRIPT = WORK_DIR / '05_scripts' / '03_ddri_prediction_modeling_feature_builder.py'

feature_spec = importlib.util.spec_from_file_location('feature_builder', FEATURE_SCRIPT)
feature_builder = importlib.util.module_from_spec(feature_spec)
sys.modules[feature_spec.name] = feature_builder
feature_spec.loader.exec_module(feature_builder)

train_spec = importlib.util.spec_from_file_location('train_eval', MODEL_SCRIPT)
train_eval = importlib.util.module_from_spec(train_spec)
sys.modules[train_spec.name] = train_eval
train_spec.loader.exec_module(train_eval)

dataset_name = 'rep15'  # 'rep15' 또는 'full161'
target_col = 'bike_change_deseasonalized'  # 또는 'bike_change_raw'

modeling_spec = next(spec for spec in feature_builder.SPECS if spec.name == dataset_name)
run_spec = train_eval.SPECS[dataset_name]

print('dataset:', dataset_name)
print('target:', target_col)
print('canonical_dir:', modeling_spec.canonical_dir)
print('modeling_dir:', modeling_spec.output_dir)
print('training_output_dir:', run_spec.output_dir)


dataset: rep15
target: bike_change_deseasonalized
canonical_dir: /Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/canonical_data
modeling_dir: /Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/modeling_data
training_output_dir: /Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/training_runs


## 3. canonical_data 입력 확인

학습은 `canonical_data`를 직접 입력으로 받는다.


In [4]:
canonical_train_path = modeling_spec.canonical_dir / 'ddri_prediction_canonical_train_2023_2024.csv'
canonical_test_path = modeling_spec.canonical_dir / 'ddri_prediction_canonical_test_2025.csv'

pd.DataFrame([
    {
        'name': 'canonical_train',
        'path': str(canonical_train_path),
        'exists': canonical_train_path.exists(),
        'size_bytes': canonical_train_path.stat().st_size if canonical_train_path.exists() else None,
    },
    {
        'name': 'canonical_test',
        'path': str(canonical_test_path),
        'exists': canonical_test_path.exists(),
        'size_bytes': canonical_test_path.stat().st_size if canonical_test_path.exists() else None,
    },
])


,name,path,exists,size_bytes
0,canonical_train,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...,True,52957254
1,canonical_test,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...,True,26328900


## 4. 정본 로드 및 연도 분할 확인


In [5]:
canonical_train = pd.read_csv(canonical_train_path)
canonical_test = pd.read_csv(canonical_test_path)
canonical_train['date'] = pd.to_datetime(canonical_train['date'])
canonical_test['date'] = pd.to_datetime(canonical_test['date'])

train_2023 = canonical_train[canonical_train['date'].dt.year == 2023].copy()
valid_2024 = canonical_train[canonical_train['date'].dt.year == 2024].copy()
test_2025 = canonical_test.copy()

pd.DataFrame([
    {'frame': 'train_2023', 'rows': len(train_2023), 'stations': train_2023['station_id'].nunique(), 'date_min': train_2023['date'].min(), 'date_max': train_2023['date'].max()},
    {'frame': 'valid_2024', 'rows': len(valid_2024), 'stations': valid_2024['station_id'].nunique(), 'date_min': valid_2024['date'].min(), 'date_max': valid_2024['date'].max()},
    {'frame': 'test_2025', 'rows': len(test_2025), 'stations': test_2025['station_id'].nunique(), 'date_min': test_2025['date'].min(), 'date_max': test_2025['date'].max()},
])


,frame,rows,stations,date_min,date_max
0,train_2023,131400,15,2023-01-01,2023-12-31
1,valid_2024,131760,15,2024-01-01,2024-12-31
2,test_2025,131400,15,2025-01-01,2025-12-31


## 5. 모델링용 전처리 적용


In [6]:
train_model = feature_builder.add_missing_flags_and_fill(train_2023)
valid_model = feature_builder.add_missing_flags_and_fill(valid_2024)
test_model = feature_builder.add_missing_flags_and_fill(test_2025)

missing_flag_cols = [c for c in train_model.columns if c.endswith('_missing')]
pd.DataFrame([
    {'frame': 'train_model', 'rows': len(train_model), 'missing_flag_cols': len(missing_flag_cols), 'target_null_rows': int(train_model[target_col].isna().sum())},
    {'frame': 'valid_model', 'rows': len(valid_model), 'missing_flag_cols': len(missing_flag_cols), 'target_null_rows': int(valid_model[target_col].isna().sum())},
    {'frame': 'test_model', 'rows': len(test_model), 'missing_flag_cols': len(missing_flag_cols), 'target_null_rows': int(test_model[target_col].isna().sum())},
])


,frame,rows,missing_flag_cols,target_null_rows
0,train_model,131400,9,0
1,valid_model,131760,9,0
2,test_model,131400,9,0


## 6. modeling_data 저장


In [7]:
modeling_spec.output_dir.mkdir(parents=True, exist_ok=True)

train_model_save = train_model.copy()
valid_model_save = valid_model.copy()
test_model_save = test_model.copy()

for frame in [train_model_save, valid_model_save, test_model_save]:
    frame['date'] = pd.to_datetime(frame['date']).dt.strftime('%Y-%m-%d')

train_model_path = modeling_spec.output_dir / 'ddri_prediction_modeling_train_2023.csv'
valid_model_path = modeling_spec.output_dir / 'ddri_prediction_modeling_valid_2024.csv'
test_model_path = modeling_spec.output_dir / 'ddri_prediction_modeling_test_2025.csv'
meta_path = modeling_spec.output_dir / 'ddri_prediction_modeling_meta.json'

train_model_save.to_csv(train_model_path, index=False, encoding='utf-8-sig')
valid_model_save.to_csv(valid_model_path, index=False, encoding='utf-8-sig')
test_model_save.to_csv(test_model_path, index=False, encoding='utf-8-sig')

modeling_meta = {
    'name': dataset_name,
    'canonical_dir': str(modeling_spec.canonical_dir),
    'output_dir': str(modeling_spec.output_dir),
    'target_primary': 'bike_change_raw',
    'target_secondary': 'bike_change_deseasonalized',
    'missing_flag_base_cols': feature_builder.MISSING_BASE_COLS,
    'train_rows': int(len(train_model_save)),
    'valid_rows': int(len(valid_model_save)),
    'test_rows': int(len(test_model_save)),
}
meta_path.write_text(json.dumps(modeling_meta, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([modeling_meta])


,name,canonical_dir,output_dir,target_primary,target_secondary,missing_flag_base_cols,train_rows,valid_rows,test_rows
0,rep15,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...,bike_change_raw,bike_change_deseasonalized,"[bike_change_lag_1, bike_change_lag_24, bike_c...",131400,131760,131400


## 7. 학습 입력 확정


In [8]:
train_fit = train_model[train_model[target_col].notna()].copy()
valid_fit = valid_model[valid_model[target_col].notna()].copy()
test_fit = test_model[test_model[target_col].notna()].copy()

feature_cols = train_eval.build_feature_cols(train_fit)
X_train = train_fit[feature_cols]
y_train = train_fit[target_col]
X_valid = valid_fit[feature_cols]
y_valid = valid_fit[target_col]
X_test = test_fit[feature_cols]
y_test = test_fit[target_col]

pd.DataFrame([
    {'frame': 'train_fit', 'rows_used': len(train_fit), 'feature_count': len(feature_cols)},
    {'frame': 'valid_fit', 'rows_used': len(valid_fit), 'feature_count': len(feature_cols)},
    {'frame': 'test_fit', 'rows_used': len(test_fit), 'feature_count': len(feature_cols)},
])


,frame,rows_used,feature_count
0,train_fit,131400,31
1,valid_fit,131760,31
2,test_fit,131400,31


## 8. 후보 모델 생성 확인


In [9]:
candidate_names = [name for name, _ in train_eval.candidate_models()]
pd.DataFrame({'candidate_model': candidate_names})


,candidate_model
0,LightGBM_RMSE
1,CatBoost_RMSE
2,StackingRegressor
3,SoftVotingRegressor


## 9. 2024 validation 모델 비교


In [10]:
selection_rows = []
for model_name, model in train_eval.candidate_models():
    model.fit(X_train, y_train)
    pred_valid = model.predict(X_valid)
    selection_rows.append({'model': model_name, **train_eval.metrics(y_valid, pred_valid)})

selection_df = pd.DataFrame(selection_rows).sort_values(['rmse', 'mae', 'r2'], ascending=[True, True, False]).reset_index(drop=True)
selection_df


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001879 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2092
[LightGBM] [Info] Number of data points in the train set: 131400, number of used features: 22
[LightGBM] [Info] Start training from score -0.000002
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001517 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2092
[LightGBM] [Info] Number of data points in the train set: 131400, number of used features: 22
[LightGBM] [Info] Start training from score -0.000002
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001231 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not e

,model,rmse,mae,r2
0,LightGBM_RMSE,0.374662,0.141453,0.924953
1,StackingRegressor,0.377858,0.143275,0.923667
2,SoftVotingRegressor,0.385698,0.147312,0.920467
3,CatBoost_RMSE,0.398519,0.151306,0.915092


## 10. 우세 모델 선택


In [11]:
best_model_name = selection_df.iloc[0]['model']
pd.DataFrame([selection_df.iloc[0].to_dict()])


,model,rmse,mae,r2
0,LightGBM_RMSE,0.374662,0.141453,0.924953


## 11. 2023+2024 재학습


In [12]:
final_X = pd.concat([X_train, X_valid], ignore_index=True)
final_y = pd.concat([y_train, y_valid], ignore_index=True)
final_model = dict(train_eval.candidate_models())[best_model_name]
final_model.fit(final_X, final_y)

pd.DataFrame([{'best_model': best_model_name, 'refit_rows': len(final_X), 'feature_count': len(feature_cols)}])


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002687 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2126
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 22
[LightGBM] [Info] Start training from score -0.000034


,best_model,refit_rows,feature_count
0,LightGBM_RMSE,263160,31


## 12. 2025 test 평가


In [13]:
test_pred = final_model.predict(X_test)
test_metrics = train_eval.metrics(y_test, test_pred)
test_metrics_df = pd.DataFrame([{'model': best_model_name, **test_metrics}])
test_metrics_df


,model,rmse,mae,r2
0,LightGBM_RMSE,0.306819,0.119461,0.941463


## 13. 결과 저장


In [14]:
run_spec.output_dir.mkdir(parents=True, exist_ok=True)

selection_path = run_spec.output_dir / f'ddri_{dataset_name}_{target_col}_selection_metrics.csv'
test_metrics_path = run_spec.output_dir / f'ddri_{dataset_name}_{target_col}_test_metrics.csv'
test_pred_path = run_spec.output_dir / f'ddri_{dataset_name}_{target_col}_test_predictions.csv'
meta_path = run_spec.output_dir / f'ddri_{dataset_name}_{target_col}_training_meta.json'

selection_df.to_csv(selection_path, index=False, encoding='utf-8-sig')
test_metrics_df.to_csv(test_metrics_path, index=False, encoding='utf-8-sig')

pred_df = test_fit[['station_id', 'date', 'hour', target_col]].copy()
pred_df['prediction'] = test_pred
pred_df.to_csv(test_pred_path, index=False, encoding='utf-8-sig')

run_meta = {
    'dataset': dataset_name,
    'target': target_col,
    'policy': {'train': '2023', 'validation': '2024', 'refit': '2023+2024', 'test': '2025'},
    'best_model': best_model_name,
    'feature_count': len(feature_cols),
    'train_rows_used': int(len(train_fit)),
    'valid_rows_used': int(len(valid_fit)),
    'test_rows_used': int(len(test_fit)),
    'selection_output': str(selection_path),
    'test_metrics_output': str(test_metrics_path),
    'test_predictions_output': str(test_pred_path),
}
meta_path.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([run_meta])


,dataset,target,policy,best_model,feature_count,train_rows_used,valid_rows_used,test_rows_used,selection_output,test_metrics_output,test_predictions_output
0,rep15,bike_change_deseasonalized,"{'train': '2023', 'validation': '2024', 'refit...",LightGBM_RMSE,31,131400,131760,131400,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...
